# Module 3 – Session 1: Practical Exercises

## Exercise 1 – The API Request

In [1]:
import requests
import pandas as pd

# 1. Define the URL
url = "https://jsonplaceholder.typicode.com/users"

# 2. Make a GET request
response = requests.get(url)

# 3. Check the status code
print("Status code:", response.status_code)

if response.status_code == 200:
    # 4. Parse JSON response into a Python list of dictionaries
    users = response.json()
    print(f"Number of users: {len(users)}\n")
    
    # 5. Extract and print name and email for each user
    print("Users (name and email):")
    for user in users:
        name = user.get("name")
        email = user.get("email")
        print(f"- {name} | {email}")
    
    # 6. Bonus: Convert to Pandas DataFrame
    df_users = pd.DataFrame(users)
    
    print("\nFirst 5 rows of the DataFrame:")
    print(df_users.head())
else:
    print("Error: Failed to fetch data from the API.")


Status code: 200
Number of users: 10

Users (name and email):
- Leanne Graham | Sincere@april.biz
- Ervin Howell | Shanna@melissa.tv
- Clementine Bauch | Nathan@yesenia.net
- Patricia Lebsack | Julianne.OConner@kory.org
- Chelsey Dietrich | Lucio_Hettinger@annie.ca
- Mrs. Dennis Schulist | Karley_Dach@jasper.info
- Kurtis Weissnat | Telly.Hoeger@billy.biz
- Nicholas Runolfsdottir V | Sherwood@rosamond.me
- Glenna Reichert | Chaim_McDermott@dana.io
- Clementina DuBuque | Rey.Padberg@karina.biz

First 5 rows of the DataFrame:
   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                                             address                  phone  \


## Exercise 2 – Web Scraper

In [5]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# 1. Define the URL
url = "https://en.wikipedia.org/wiki/List_of_largest_companies_by_revenue"

# 2. Fetch the HTML content
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("Status code:", response.status_code)

if response.status_code != 200:
    raise Exception("Failed to load page")

html_content = response.text

# 3. Create BeautifulSoup object
soup = BeautifulSoup(html_content, "html.parser")

# 4. Find the main table (with class 'wikitable')
# Often the main table is the first 'wikitable' on the page.
table = soup.find("table", {"class": "wikitable"})

if table is None:
    raise Exception("Could not find the main 'wikitable' on the page")

# 5. Extract table header
header_row = table.find("tr")
header_cells = header_row.find_all(["th", "td"])

columns = [cell.get_text(strip=True) for cell in header_cells]

print("Columns:", columns)

# 6. Extract table body rows
data_rows = []
for row in table.find_all("tr")[1:]:  # skip header row
    cells = row.find_all(["th", "td"])
    # Skip empty rows
    if len(cells) == 0:
        continue
    row_data = [cell.get_text(strip=True) for cell in cells]
    data_rows.append(row_data)

print(f"Number of data rows: {len(data_rows)}")

# 7. Create Pandas DataFrame
df_companies = pd.DataFrame(data_rows, columns=columns)

# 8. Display the DataFrame (or its head)
df_cleaned = df_companies.iloc[1:].reset_index(drop=True)

df_cleaned.head()


Status code: 200
Columns: ['Ranks', 'Name', 'Industry', 'Revenue', 'Profit', 'Employees', 'Headquarters[note 1]', 'State-owned', 'Ref.']
Number of data rows: 51


,Ranks,Name,Industry,Revenue,Profit,Employees,Headquarters[note 1],State-owned,Ref.
0,1,Walmart,Retail,"$680,985","$19,436","2,100,000",United States,,[1]
1,2,Amazon,Retailinformation technology,"$637,959","$59,248","1,556,000",,[4],None
2,3,State Grid Corporation of China,Electricity,"$545,948","$9,204","1,361,423",China,,[5]
3,4,Saudi Aramco,Oil and gas,"$480,446","$106,246","73,311",Saudi Arabia,,[6]
4,5,China Petrochemical Corporation,"$429,700","$9,393","513,434",China,,[7],None


## Exercise 3 – Strategy: Daily Weather Data for Yerevan

**1. What would be your first approach: API or scraping? Why?**

My first approach would be to look for a weather API (for example, OpenWeatherMap or similar services) and try to get the data via an API. The reason is that APIs usually provide structured, predictable, and documented data (JSON), which is easier to automate and integrate into Python scripts. Also, APIs are designed for programmatic access, so there is a lower risk that the structure will frequently change.


**2. Two potential problems with the API route**

1. Many APIs have usage limits (rate limits) or require a paid plan, especially if we want to make requests every day for a whole year.
2. The data provided by the API may not perfectly match our needs; for example, the level of detail, update frequency, or access to historical data may be limited.


**3. Two potential problems with the web scraping route**

1. The website’s HTML structure may change at any time, which would break our scraper and require constant maintenance and updates.
2. Some websites forbid automated scraping in their Terms of Service or technically restrict it (e.g., blocking, Captcha, rate limiting), which can create legal or technical issues.
